# 🏙️ Airbnb NYC — Exploratory Data Analysis & Visualization

**Dataset:** [NYC Airbnb Open Data – Kaggle](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data)

**Goal:** Analyze 49,000+ Airbnb listings across New York City to uncover pricing patterns, neighborhood trends, and host behavior using Python visualization libraries.

**Libraries used:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `plotly`

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install libraries (only needed in Colab)
!pip install plotly --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

print('✅ All libraries loaded successfully!')

## 📂 Step 2: Load the Dataset

> Download `AB_NYC_2019.csv` from [Kaggle](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) and upload it here, OR run the cell below to load it directly via URL.

In [ ]:
# Load dataset directly from a public URL (no download needed)
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/AB_NYC_2019.csv'
df = pd.read_csv(url)

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

## 🔍 Step 3: Data Overview & Cleaning

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Data Cleaning
# Fill missing text columns
df['name'] = df['name'].fillna('No Name')
df['host_name'] = df['host_name'].fillna('Unknown')

# Fill missing reviews with 0
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
df['last_review'] = df['last_review'].fillna('Never')

# Remove extreme price outliers (keep listings $10–$1000/night)
df = df[(df['price'] >= 10) & (df['price'] <= 1000)]

print(f'✅ Cleaned dataset shape: {df.shape}')
print('\n=== Basic Statistics ===')
df[['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'availability_365']].describe().round(2)

## 📊 Step 4: Visualizations

### 4.1 — Distribution of Listings by Neighbourhood Group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
neighbourhood_counts = df['neighbourhood_group'].value_counts()
sns.barplot(x=neighbourhood_counts.index, y=neighbourhood_counts.values,
            palette='Blues_d', ax=axes[0])
axes[0].set_title('Number of Listings per Borough', fontweight='bold')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Number of Listings')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(neighbourhood_counts.values, labels=neighbourhood_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Blues_d', len(neighbourhood_counts)))
axes[1].set_title('Share of Listings by Borough', fontweight='bold')

plt.suptitle('Airbnb Listings Distribution across NYC Boroughs', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.2 — Price Distribution by Borough

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of overall price distribution
axes[0].hist(df['price'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['price'].median(), color='red', linestyle='--', linewidth=2, label=f"Median: ${df['price'].median():.0f}")
axes[0].axvline(df['price'].mean(), color='orange', linestyle='--', linewidth=2, label=f"Mean: ${df['price'].mean():.0f}")
axes[0].set_title('Overall Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price per Night ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot by borough
order = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='neighbourhood_group', y='price', order=order,
            palette='Set2', ax=axes[1], showfliers=False)
axes[1].set_title('Price Distribution by Borough (outliers hidden)', fontweight='bold')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Price per Night ($)')

plt.suptitle('Airbnb Price Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 — Room Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count of room types
room_counts = df['room_type'].value_counts()
colors = ['#4C72B0', '#DD8452', '#55A868']
sns.barplot(x=room_counts.index, y=room_counts.values, palette=colors, ax=axes[0])
axes[0].set_title('Listings by Room Type', fontweight='bold')
axes[0].set_xlabel('Room Type')
axes[0].set_ylabel('Number of Listings')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Avg price by room type
avg_price = df.groupby('room_type')['price'].mean().sort_values(ascending=False)
sns.barplot(x=avg_price.index, y=avg_price.values, palette=colors, ax=axes[1])
axes[1].set_title('Average Price by Room Type', fontweight='bold')
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Average Price ($)')
for p in axes[1].patches:
    axes[1].annotate(f'${p.get_height():.0f}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

plt.suptitle('Room Type Breakdown', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4 — Top 10 Neighbourhoods by Average Price

In [ ]:
top_neighbourhoods = (df.groupby('neighbourhood')['price']
                        .mean()
                        .sort_values(ascending=False)
                        .head(10)
                        .reset_index())
top_neighbourhoods.columns = ['Neighbourhood', 'Avg Price']

plt.figure(figsize=(12, 6))
bars = plt.barh(top_neighbourhoods['Neighbourhood'][::-1],
                top_neighbourhoods['Avg Price'][::-1],
                color=sns.color_palette('RdYlGn', 10)[::-1])

for bar, val in zip(bars, top_neighbourhoods['Avg Price'][::-1]):
    plt.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
             f'${val:.0f}', va='center', fontsize=10)

plt.title('Top 10 Most Expensive Neighbourhoods (Avg Price/Night)', fontsize=14, fontweight='bold')
plt.xlabel('Average Price ($)')
plt.tight_layout()
plt.show()

### 4.5 — Correlation Heatmap

In [ ]:
corr_cols = ['price', 'minimum_nights', 'number_of_reviews',
             'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True,
            annot_kws={'size': 11})
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.6 — Price vs. Number of Reviews (Interactive Scatter — Plotly)

In [ ]:
sample = df[df['number_of_reviews'] > 0].sample(3000, random_state=42)

fig = px.scatter(
    sample,
    x='number_of_reviews',
    y='price',
    color='neighbourhood_group',
    size='availability_365',
    hover_data=['name', 'room_type', 'neighbourhood'],
    title='Price vs. Number of Reviews (Interactive)',
    labels={'number_of_reviews': 'Number of Reviews', 'price': 'Price per Night ($)',
            'neighbourhood_group': 'Borough'},
    color_discrete_sequence=px.colors.qualitative.Set2,
    opacity=0.7,
    template='plotly_white'
)
fig.update_layout(title_font_size=16)
fig.show()

### 4.7 — Interactive Geographic Map of Listings (Plotly)

In [ ]:
map_sample = df.sample(5000, random_state=42)

fig = px.scatter_mapbox(
    map_sample,
    lat='latitude',
    lon='longitude',
    color='price',
    size='price',
    hover_name='name',
    hover_data={'neighbourhood_group': True, 'room_type': True,
                'price': True, 'latitude': False, 'longitude': False},
    color_continuous_scale='Viridis',
    range_color=[10, 400],
    size_max=15,
    zoom=10,
    center={'lat': 40.7128, 'lon': -74.0060},
    mapbox_style='carto-positron',
    title='NYC Airbnb Listings Map — Colored by Price',
    labels={'price': 'Price/Night ($)'}
)
fig.update_layout(title_font_size=16, height=600)
fig.show()

### 4.8 — Availability Heatmap by Borough & Room Type

In [ ]:
pivot = df.pivot_table(values='availability_365',
                        index='neighbourhood_group',
                        columns='room_type',
                        aggfunc='mean').round(1)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, annot_kws={'size': 12})
plt.title('Average Availability (Days/Year) by Borough & Room Type', fontsize=14, fontweight='bold')
plt.xlabel('Room Type')
plt.ylabel('Borough')
plt.tight_layout()
plt.show()

## 📝 Step 5: Key Insights

After analyzing 49,000+ Airbnb listings across NYC, here are the main findings:

1. **Manhattan dominates** in both listing count and price — median price significantly higher than other boroughs.
2. **Entire home/apt listings** cost on average 2–3x more than private rooms.
3. **Staten Island and the Bronx** have the most available listings year-round, suggesting lower demand.
4. **Price and number of reviews** show a weak negative correlation — cheaper listings tend to get reviewed more.
5. **Top neighbourhoods** like Fort Wadsworth and Woodrow command premium prices despite being outside Manhattan.

---
*Project by: [Your Name] | Dataset: NYC Airbnb Open Data (Kaggle) | Tools: Python, Pandas, Seaborn, Plotly*